# Aprendizado de Máquina — Aula prática 05

## Árvores de Regressão e Ensembles

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

O KNN falha em dimensão alta porque trata
todas as covariáveis igualmente. Precisamos de um método que **escolha em quais
direções olhar**. A árvore de regressão é exatamente isso — e vem com um defeito
enorme de fábrica.

> **A árvore tem viés baixo e variância altíssima. Esta aula inteira é sobre o que
> fazer com essa variância.**

São três respostas, em ordem histórica e em ordem de qualidade: podar (jogar fora
complexidade), ensacar (fazer média de muitas) e impulsionar (construir devagar,
corrigindo o próprio erro). Vamos medir as três, e medir também o **piso** que a
segunda delas não consegue furar — o número que explica por que as florestas
aleatórias existem.

### Objetivos

Ao final deste notebook você deve ser capaz de:

- ler uma árvore nos dois formatos — partição do plano e diagrama — e reconhecer
  onde ela captura uma interação;
- exibir um caso em que a busca **gananciosa** não enxerga a estrutura dos dados;
- podar por custo–complexidade e escolher $\alpha$ por validação cruzada;
- medir a **instabilidade** de uma árvore e a **correlação** $\rho$ entre árvores
  de um mesmo *ensemble*;
- confirmar numericamente o piso $\rho\,v(x)$ da variância, e ver a floresta
  aleatória abaixá-lo;
- usar a estimativa OOB e conferi-la contra um conjunto de teste;
- reconhecer que o excesso de árvores é inofensivo na floresta e caro no
  *boosting*;
- desconfiar de `feature_importances_` — e saber o que usar no lugar.

---
## 1. Importando os pacotes

In [ ]:
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots

Os objetos novos são a árvore e os três *ensembles* do `scikit-learn`, mais o
`plot_tree` (que desenha a árvore) e o `permutation_importance` (que aparece na
Seção 10, quando a importância padrão nos decepcionar).

In [ ]:
import sklearn.linear_model as skl
import sklearn.model_selection as skm
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import (BaggingRegressor, GradientBoostingRegressor,
                              RandomForestRegressor)
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
import warnings
warnings.filterwarnings("ignore")

---
## 2. A árvore é uma partição

Uma árvore parte o espaço de covariáveis em retângulos e prevê, em cada um, a
média das respostas de treino que caíram ali. As duas representações — o diagrama
e a partição — são a mesma coisa vista de dois ângulos.

Vamos construir um problema em $[0,10]^2$ com uma **interação**: o efeito de $x_2$
depende de onde está $x_1$. Nenhum modelo linear pega isso sem que alguém escreva
o termo de interação à mão.

In [ ]:
rng = np.random.default_rng(0)
n = 300
X2 = rng.uniform(0, 10, size=(n, 2))


def alvo_2d(M):
    esquerda = M[:, 0] <= 4
    return np.where(esquerda,
                    np.where(M[:, 1] <= 6, 3.0, 4.5),      # a esquerda, corte em 6
                    np.where(M[:, 1] <= 3, 2.5, 0.5))      # a direita, corte em 3


y2 = alvo_2d(X2) + rng.normal(0, 0.35, size=n)

arvore = DecisionTreeRegressor(max_depth=2, random_state=0).fit(X2, y2)
print(f"R^2 no treino: {arvore.score(X2, y2):.3f}")

Antes de olhar o que a árvore encontrou, vale ver o que ela está procurando.

A função `alvo_2d` é **constante por partes em quatro retângulos** — exatamente a
forma que uma árvore sabe representar. A interação anunciada acima está nos
números: o corte de $x_2$ vale $6$ à esquerda de $x_1 = 4$ e $3$ à direita.

O ruído somado tem desvio-padrão $0{,}35$, contra saltos entre regiões vizinhas
que vão de $0{,}5$ a $4{,}0$ — folga suficiente para a árvore reencontrar os
cortes, como a próxima célula mostra.

In [ ]:
gx_r, gy_r = np.meshgrid(np.linspace(0, 10, 300), np.linspace(0, 10, 300))
r_verdade = alvo_2d(np.c_[gx_r.ravel(), gy_r.ravel()]).reshape(gx_r.shape)

fig, ax = subplots(figsize=(5.0, 4.0))
mapa = ax.pcolormesh(gx_r, gy_r, r_verdade, cmap="viridis", shading="auto",
                     vmin=0.5, vmax=4.5)
ax.plot([4, 4], [0, 10], color="white", lw=2)     # o corte de x1
ax.plot([0, 4], [6, 6], color="white", lw=2)      # o corte de x2, a esquerda
ax.plot([4, 10], [3, 3], color="white", lw=2)     # o corte de x2, a direita
for px, py, valor, cor in [(2, 3, 3.0, "white"), (2, 8, 4.5, "black"),
                           (7, 1.5, 2.5, "white"), (7, 6.5, 0.5, "white")]:
    ax.text(px, py, f"{valor}", color=cor, ha="center", va="center", fontsize=11)
ax.set_xlabel("x1"); ax.set_ylabel("x2")
ax.set_title("r(x): a funcao que a arvore vai procurar", fontsize=9)
fig.colorbar(mapa, ax=ax, label="r(x)")

In [ ]:
fig, (ax1, ax2) = subplots(1, 2, figsize=(10, 3.8),
                           gridspec_kw={"width_ratios": [1, 1.35]})

gx, gy = np.meshgrid(np.linspace(0, 10, 300), np.linspace(0, 10, 300))
Z = arvore.predict(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
ax1.contourf(gx, gy, Z, levels=20, cmap="viridis", alpha=0.65)
ax1.scatter(X2[:, 0], X2[:, 1], c=y2, s=10, cmap="viridis", edgecolor="k", lw=0.2)
ax1.set_xlabel("x1"); ax1.set_ylabel("x2")
ax1.set_title("a particao do plano", fontsize=9)

plot_tree(arvore, ax=ax2, feature_names=["x1", "x2"], filled=True,
          rounded=True, precision=2, fontsize=7)
ax2.set_title("a mesma coisa, como arvore", fontsize=9)

In [ ]:
t = arvore.tree_
print("corte da raiz          : x%d <= %.2f" % (t.feature[0] + 1, t.threshold[0]))
print("corte do filho esquerdo: x%d <= %.2f" % (t.feature[1] + 1, t.threshold[1]))
print("corte do filho direito : x%d <= %.2f" % (t.feature[4] + 1, t.threshold[4]))

Repare no essencial: os dois filhos cortam **a mesma variável $x_2$, em pontos
diferentes**. É assim que uma árvore representa uma interação — sem que ninguém a
tenha especificado, sem termo de produto, sem nada. Foi só deixar cada ramo
escolher o próprio corte.

Daí saem as quatro virtudes estruturais das árvores: são interpretáveis, aceitam
covariáveis categóricas sem *dummies*, fazem seleção de variáveis sozinhas e
capturam interações de graça.

E a desvantagem estrutural, que também é geométrica: **todo corte é perpendicular
a um eixo**. Uma fronteira diagonal como $x_1 + x_2 > 10$ obriga a árvore a
construir uma escadinha, e cada degrau custa dados. Se você suspeita de relação
linear, um modelo linear faz melhor com muito menos.

---
## 3. A miopia do algoritmo ganancioso

Encontrar a árvore ótima é NP-difícil, então o algoritmo escolhe, a cada passo, o
corte que **mais reduz o RSS agora** — e nunca volta atrás. Isso tem um custo, e
dá para exibi-lo com um exemplo de quatro linhas.

Considere $y = \operatorname{sinal}(x_1 x_2)$: vale $+1$ nos quadrantes ímpares e
$-1$ nos pares. **Quatro regiões resolvem o problema exatamente.** Mas repare no
que acontece com o *primeiro* corte.

In [ ]:
rng_x = np.random.default_rng(1)
Xx = rng_x.uniform(-1, 1, size=(2000, 2))
yx = np.sign(Xx[:, 0] * Xx[:, 1])
Xx_te = rng_x.uniform(-1, 1, size=(20_000, 2))
yx_te = np.sign(Xx_te[:, 0] * Xx_te[:, 1])


def r2_teste(pred):
    return 1 - ((yx_te - pred) ** 2).sum() / ((yx_te - yx_te.mean()) ** 2).sum()


rss_total = ((yx - yx.mean()) ** 2).sum()
melhor = 0.0
for j in (0, 1):
    for t_corte in np.linspace(-0.9, 0.9, 181):
        esq = Xx[:, j] <= t_corte
        if esq.sum() < 5 or (~esq).sum() < 5:
            continue
        rss = (((yx[esq] - yx[esq].mean()) ** 2).sum()
               + ((yx[~esq] - yx[~esq].mean()) ** 2).sum())
        melhor = max(melhor, rss_total - rss)

print(f"RSS sem nenhum corte        : {rss_total:.1f}")
print(f"melhor reducao com UM corte : {melhor:.4f}  ({100*melhor/rss_total:.3f}%)")

quadrante_tr = (Xx[:, 0] > 0).astype(int) * 2 + (Xx[:, 1] > 0).astype(int)
quadrante_te = (Xx_te[:, 0] > 0).astype(int) * 2 + (Xx_te[:, 1] > 0).astype(int)
medias = np.array([yx[quadrante_tr == q].mean() for q in range(4)])
print(f"\nR^2 no teste da particao em 4 quadrantes (o oraculo): "
      f"{r2_teste(medias[quadrante_te]):.4f}")

In [ ]:
print("arvore gananciosa, R^2 no TESTE:")
for prof in (1, 2, 3, 4, 6, 8, None):
    m = DecisionTreeRegressor(max_depth=prof, random_state=0).fit(Xx, yx)
    rot = "cheia" if prof is None else str(prof)
    print(f"   profundidade {rot:>5}  ({m.get_n_leaves():3d} folhas): "
          f"{r2_teste(m.predict(Xx_te)):.4f}")

parada = DecisionTreeRegressor(min_impurity_decrease=0.005, random_state=0).fit(Xx, yx)
print(f"\ncom parada precoce (min_impurity_decrease=0,005): "
      f"{parada.get_n_leaves()} folha(s), R^2 = {r2_teste(parada.predict(Xx_te)):.4f}")

O melhor primeiro corte não reduz **nada**: qualquer reta vertical ou horizontal
deixa os dois lados com média zero. E é isso que a última linha mostra: um critério
de parada do tipo *"só divida se a redução for relevante"* — um `min_impurity_decrease`
modestíssimo — **mata a árvore na raiz**, deixando-a com uma única folha e $R^2$
zero, num problema que quatro folhas resolvem perfeitamente.

A árvore gananciosa acaba chegando lá, mas pelo caminho longo. Nas profundidades 1
a 4 ela não sai do lugar; dá um salto na 6; e só na 8 alcança o oráculo — com
**12 folhas**, o triplo do necessário, porque cada corte mal colocado no topo tem
de ser remendado lá embaixo.

Duas consequências práticas, e as duas organizam o resto da aula:

1. **Cresça demais e pode depois**, em vez de parar cedo. A poda pode reconhecer
   *a posteriori* que sobrava estrutura; um critério de parada precoce decide antes
   de ter visto o que havia adiante.
2. **Uma árvore só não é o produto final.** Se a decisão no topo é praticamente um
   empate, ela é decidida por ruído — e a próxima seção mede exatamente isso.

Em duas dimensões, quatro regiões resolviam o problema e o primeiro corte não via
nada. Em três, oito regiões resolvem — e a pergunta é se a miopia piora ou melhora
quando há mais um eixo para o algoritmo tropeçar.

In [ ]:
rng_3 = np.random.default_rng(1)
X3 = rng_3.uniform(-1, 1, size=(2000, 3))
y3 = np.sign(X3[:, 0] * X3[:, 1] * X3[:, 2])
X3_te = rng_3.uniform(-1, 1, size=(20_000, 3))
y3_te = np.sign(X3_te[:, 0] * X3_te[:, 1] * X3_te[:, 2])


def r2_3d(pred):
    return 1 - ((y3_te - pred) ** 2).sum() / ((y3_te - y3_te.mean()) ** 2).sum()


def melhor_corte(Xc, yc):
    """Maior reducao de RSS entre todos os cortes de todas as colunas."""
    rss_pai = ((yc - yc.mean()) ** 2).sum()
    melhor_red = 0.0
    for j in range(Xc.shape[1]):
        for t in np.linspace(-0.9, 0.9, 181):
            esq = Xc[:, j] <= t
            if esq.sum() < 5 or (~esq).sum() < 5:
                continue
            rss = (((yc[esq] - yc[esq].mean()) ** 2).sum()
                   + ((yc[~esq] - yc[~esq].mean()) ** 2).sum())
            melhor_red = max(melhor_red, rss_pai - rss)
    return rss_pai, melhor_red


rss_pai, red1 = melhor_corte(X3, y3)
print(f"1o corte: RSS {rss_pai:.1f}, melhor reducao {red1:.4f} ({100*red1/rss_pai:.3f}%)")

dentro = X3[:, 0] <= 0                      # ja dentro do melhor primeiro corte
rss_f, red2 = melhor_corte(X3[dentro], y3[dentro])
print(f"2o corte: RSS {rss_f:.1f}, melhor reducao {red2:.4f} ({100*red2/rss_f:.3f}%)")

print("\narvore gananciosa, R^2 no TESTE:")
for prof in (1, 2, 3, 4, 5, 6, 8, None):
    m3 = DecisionTreeRegressor(max_depth=prof, random_state=0).fit(X3, y3)
    rot = "cheia" if prof is None else str(prof)
    print(f"   profundidade {rot:>5}  ({m3.get_n_leaves():3d} folhas): {r2_3d(m3.predict(X3_te)):+.4f}")

parada3 = DecisionTreeRegressor(min_impurity_decrease=0.005, random_state=0).fit(X3, y3)
print(f"\ncom parada precoce: {parada3.get_n_leaves()} folha(s), "
      f"R^2 = {r2_3d(parada3.predict(X3_te)):+.4f}")

**Piora, e piora de todos os jeitos.**

O primeiro corte reduz o RSS em $0{,}194\%$ e o segundo em $0{,}350\%$ — ou seja, a
cegueira não é só do primeiro passo. Mesmo depois de fixar $x_1 \le 0$, os dois
quadrantes que sobram continuam se cancelando dois a dois, e o segundo corte
também não vê nada. Em duas dimensões bastava um corte para o problema virar
resolvível; em três, são necessários **dois** cortes às cegas antes de qualquer
sinal aparecer.

A consequência está na tabela de profundidades. Oito regiões resolvem o problema
exatamente, o que caberia numa árvore de profundidade 3. A árvore gananciosa tem
$R^2$ **negativo** até a profundidade 5 — pior que chutar a média —, encosta em
$0{,}17$ na 6 e só chega a $0{,}87$ com a árvore cheia, de 59 folhas. Ela precisa
gastar cinco níveis de cortes inúteis antes de os quadrantes ficarem pequenos o
bastante para o sinal emergir.

E a parada precoce, que no caso de duas dimensões já era perigosa, aqui é fatal:
com `min_impurity_decrease=0,005` a árvore fica com **uma folha**. O critério
nunca vê $0{,}5\%$ de redução em lugar nenhum, e desiste antes de começar.

É por isso que a poda por complexidade de custo, que cresce a árvore inteira e
*depois* corta, é o método padrão — e não a parada precoce.

---
## 4. Crescer e podar

A poda por custo–complexidade minimiza

$$\sum_{k}\sum_{i:\,X_i \in R_k}(Y_i - \widehat y_{R_k})^2 + \alpha\,|T|,$$

onde $|T|$ é o número de folhas. É a mesma estrutura da Aula 02 — ajuste mais
$\lambda \times$ complexidade — com o número de folhas no papel da norma do vetor
de coeficientes.

O `scikit-learn` entrega a sequência inteira de $\alpha$ em que a árvore ótima
muda, de graça.

In [ ]:
rng_p = np.random.default_rng(5)
n_p, d_p = 400, 6


def alvo_p(M):
    return np.sin(1.5 * M[:, 0]) + 0.8 * M[:, 1] * M[:, 2] + 0.5 * M[:, 0] ** 2


X_p = rng_p.uniform(-2, 2, size=(n_p, d_p))
y_p = alvo_p(X_p) + rng_p.normal(0, 1.0, size=n_p)
X_pte = rng_p.uniform(-2, 2, size=(4000, d_p))
r_pte = alvo_p(X_pte)

cheia = DecisionTreeRegressor(random_state=0).fit(X_p, y_p)
caminho = cheia.cost_complexity_pruning_path(X_p, y_p)
alphas = caminho.ccp_alphas[:-1]          # o ultimo colapsa para a raiz
print(f"a arvore cheia tem {cheia.get_n_leaves()} folhas")
print(f"o caminho de poda tem {len(alphas)} valores de alpha, "
      f"de {alphas.min():.5f} a {alphas.max():.3f}")

In [ ]:
folhas, risco_te = [], []
for a in alphas:
    m = DecisionTreeRegressor(ccp_alpha=a, random_state=0).fit(X_p, y_p)
    folhas.append(m.get_n_leaves())
    risco_te.append(np.mean((m.predict(X_pte) - r_pte) ** 2))

busca = skm.GridSearchCV(DecisionTreeRegressor(random_state=0),
                         {"ccp_alpha": alphas},
                         cv=skm.KFold(5, shuffle=True, random_state=0),
                         scoring="neg_mean_squared_error").fit(X_p, y_p)
a_cv = busca.best_params_["ccp_alpha"]
m_cv = DecisionTreeRegressor(ccp_alpha=a_cv, random_state=0).fit(X_p, y_p)

fig, (ax1, ax2) = subplots(1, 2, figsize=(7.8, 3.0))
ax1.plot(alphas, folhas, drawstyle="steps-post", color="steelblue")
ax1.axvline(a_cv, ls=":", color="green")
ax1.set_xscale("log"); ax1.set_yscale("log")
ax1.set_xlabel("alpha"); ax1.set_ylabel("numero de folhas")
ax1.set_title("podar e' andar para a direita", fontsize=9)

ax2.plot(alphas, risco_te, color="crimson")
ax2.axvline(a_cv, ls=":", color="green", label="alpha escolhido por CV")
ax2.set_xscale("log")
ax2.set_xlabel("alpha"); ax2.set_ylabel("risco (teste)")
ax2.set_title("e existe um alpha bom", fontsize=9)
ax2.legend(fontsize=8)

print(f"alpha escolhido por CV : {a_cv:.4f}")
print(f"folhas depois da poda  : {m_cv.get_n_leaves()}  (eram {cheia.get_n_leaves()})")
print(f"risco da arvore cheia  : {np.mean((cheia.predict(X_pte) - r_pte) ** 2):.4f}")
print(f"risco da arvore podada : {np.mean((m_cv.predict(X_pte) - r_pte) ** 2):.4f}")

---
## 5. A instabilidade, medida

"Árvores têm variância alta" costuma ser dito e raramente mostrado. O mecanismo é
específico: se o **corte da raiz** muda, toda a estrutura abaixo dele muda junto.
Vamos reamostrar os dados e olhar a raiz.

In [ ]:
rng_b = np.random.default_rng(9)
raizes, cortes = [], []
for _ in range(200):
    idx = rng_b.integers(0, n_p, n_p)          # reamostra com reposicao
    m = DecisionTreeRegressor(max_depth=4, random_state=0).fit(X_p[idx], y_p[idx])
    raizes.append(int(m.tree_.feature[0]))
    cortes.append(float(m.tree_.threshold[0]))

contagem = pd.Series(raizes).value_counts().sort_index()
contagem.index = [f"x{j+1}" for j in contagem.index]
print("variavel escolhida na RAIZ, em 200 reamostragens:")
print(contagem.to_string())
print(f"\nponto de corte: de {min(cortes):.2f} a {max(cortes):.2f} "
      f"(desvio-padrao {np.std(cortes):.2f}, num dominio de largura 4)")

fig, ax = subplots(figsize=(5.2, 2.6))
ax.hist(cortes, bins=30, color="steelblue")
ax.set_xlabel("ponto de corte da raiz"); ax.set_ylabel("frequencia")
ax.set_xlim(-2, 2)

Aqui a instabilidade tem uma forma específica, e vale ler com atenção porque ela
prepara a próxima seção.

A **variável** da raiz não muda nunca: é sempre $x_1$, nas 200 reamostragens. Não
é acaso — na nossa população $x_1$ entra duas vezes ($\sin(1{,}5x_1)$ e
$0{,}5x_1^2$), enquanto as outras entram só pela interação $x_2x_3$. Existe uma
covariável dominante, e todas as árvores a agarram.

O **ponto de corte**, esse passeia por quase um quarto do domínio. E como tudo
abaixo da raiz depende de onde ela cortou, cada reamostragem produz uma árvore
substancialmente diferente. É variância pura.

Guarde a primeira observação: *uma covariável dominante faz todas as árvores
começarem igual*. É exatamente essa a correlação que a próxima seção vai medir — e
que a floresta aleatória foi inventada para quebrar.

---
## 6. Por que a média ajuda — e onde ela para

A Proposição da aula diz: se $g_1,\dots,g_B$ são não-viesados, de mesma variância
$v(x)$ e **não-correlacionados**, então $\Var(\bar g(x)) = v(x)/B$, e a média nunca
piora. A hipótese de não-correlação é forte demais para ser verdade, e a conta
honesta é

$$\Var(\bar g(x)) = \rho\,v(x) + \frac{1-\rho}{B}\,v(x)
  \ \xrightarrow[B\to\infty]{}\ \rho\,v(x).$$

Por mais árvores que você acrescente, a variância **não desce abaixo de
$\rho\,v(x)$**. Só o segundo termo morre. Isso é mensurável, e vale medir: é o
número que justifica a existência das florestas aleatórias.

O plano: sortear 120 conjuntos de treino independentes; em cada um, crescer **duas**
árvores com sementes diferentes; e olhar, ponto de teste a ponto de teste, a
correlação entre as duas ao longo dos 120 conjuntos. Isso é $\rho$. A variância de
uma delas é $v$.

In [ ]:
def medir_rho_e_v(max_features, n_conj=120, B_lista=(1, 5, 25, 100), semente=13):
    rng_m = np.random.default_rng(semente)
    X_ref = rng_m.uniform(-2, 2, size=(400, d_p))       # pontos de teste fixos
    r_ref = alvo_p(X_ref)

    par = np.zeros((2, n_conj, len(X_ref)))
    ens = {B: np.zeros((n_conj, len(X_ref))) for B in B_lista}
    for c in range(n_conj):
        Xc = rng_m.uniform(-2, 2, size=(n_p, d_p))
        yc = alvo_p(Xc) + rng_m.normal(0, 1.0, size=n_p)
        for s in (0, 1):
            par[s, c] = RandomForestRegressor(
                n_estimators=1, max_features=max_features, bootstrap=True,
                random_state=100 * c + s).fit(Xc, yc).predict(X_ref)
        for B in B_lista:
            ens[B][c] = RandomForestRegressor(
                n_estimators=B, max_features=max_features,
                random_state=7).fit(Xc, yc).predict(X_ref)

    v = par[0].var(axis=0).mean()
    rho = np.mean([np.corrcoef(par[0][:, j], par[1][:, j])[0, 1]
                   for j in range(len(X_ref))])
    var_ens = {B: ens[B].var(axis=0).mean() for B in B_lista}
    return rho, v, var_ens


rho_bag, v_bag, var_bag = medir_rho_e_v(max_features=1.0)      # bagging: usa tudo
rho_rf, v_rf, var_rf = medir_rho_e_v(max_features=1 / 3)       # floresta
print(f"bagging  (max_features=1  ): rho = {rho_bag:.3f}   v = {v_bag:.4f}")
print(f"floresta (max_features=1/3): rho = {rho_rf:.3f}   v = {v_rf:.4f}")

In [ ]:
Bs = np.array([1, 5, 25, 100])
linhas = []
for nome, rho, v, var_ens in [("bagging", rho_bag, v_bag, var_bag),
                              ("floresta", rho_rf, v_rf, var_rf)]:
    for B in Bs:
        linhas.append({"ensemble": nome, "B": B,
                       "variancia medida": var_ens[B],
                       "rho*v + (1-rho)*v/B": rho * v + (1 - rho) * v / B,
                       "piso rho*v": rho * v})
tab = pd.DataFrame(linhas).set_index(["ensemble", "B"])
tab.round(4)

In [ ]:
fig, ax = subplots(figsize=(5.4, 3.2))
Bfino = np.logspace(0, 3, 100)
for nome, rho, v, var_ens, cor in [("bagging", rho_bag, v_bag, var_bag, "crimson"),
                                   ("floresta", rho_rf, v_rf, var_rf, "steelblue")]:
    ax.plot(Bs, [var_ens[B] for B in Bs], "o", ms=6, color=cor, label=f"{nome} (medido)")
    ax.plot(Bfino, rho * v + (1 - rho) * v / Bfino, color=cor, lw=1.2,
            label=f"{nome} (formula)")
    ax.axhline(rho * v, ls=":", color=cor, lw=1)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("B (numero de arvores)"); ax.set_ylabel("variancia da predicao")
ax.set_title("as linhas pontilhadas sao os pisos rho*v", fontsize=9)
ax.legend(fontsize=7.5)

> **A lição.** A fórmula acerta os pontos medidos em toda a faixa, e as duas curvas
> achatam em **patamares diferentes**. O $\rho$ do *bagging* é quase o dobro do da
> floresta, e é a Seção 5 explicando: com uma covariável dominante, todas as
> árvores começam no mesmo lugar e ficam parecidas.
>
> A floresta ataca exatamente esse termo. Sorteando $m < p$ covariáveis **em cada
> nó**, ela obriga uma fração $1 - m/p$ dos nós a olhar para outra coisa, e o piso
> cai junto.
>
> Note o que *não* melhora: $v$, a variância de uma árvore isolada, é **maior** na
> floresta — cada árvore individual é pior, porque às vezes é obrigada a usar uma
> variável ruim. A floresta troca qualidade individual por diversidade, e o negócio
> compensa porque quem manda no limite é o produto $\rho v$.
>
> E repare no que a tabela diz sobre $B$: entre $B=25$ e $B=100$ a variância quase
> não se move, porque a essa altura o termo $(1-\rho)v/B$ já é pequeno perto do
> piso. Acrescentar árvores depois disso é gastar computação para nada.

---
## 7. *Bagging*, OOB e a conta dos 37%

Cada amostra bootstrap deixa de fora, em média, $(1-1/n)^n \to e^{-1} \approx 37\%$
das observações. Elas são as *out-of-bag*, e servem para estimar o risco **de
graça**: cada observação é prevista só pelas árvores que não a viram.

In [ ]:
bag = BaggingRegressor(DecisionTreeRegressor(random_state=0), n_estimators=300,
                       oob_score=True, random_state=0, n_jobs=-1).fit(X_p, y_p)

# o oob_score_ do sklearn e' um R^2; convertemos para EQM para comparar
eqm_oob = (1 - bag.oob_score_) * ((y_p - y_p.mean()) ** 2).mean()
y_pte = r_pte + rng_p.normal(0, 1.0, size=len(r_pte))

print(f"EQM estimado por OOB          : {eqm_oob:.4f}")
print(f"EQM medido num teste de 4000  : {mean_squared_error(y_pte, bag.predict(X_pte)):.4f}")
print(f"EQM de UMA arvore (teste)     : "
      f"{mean_squared_error(y_pte, cheia.predict(X_pte)):.4f}")

A estimativa OOB chega perto do teste — e não custou nem uma dobra de validação
cruzada. Vale conferir também a conta que a sustenta:

In [ ]:
rng_o = np.random.default_rng(3)
for n_teste in (50, 400, 5000):
    sorteios = rng_o.integers(0, n_teste, size=(3000, n_teste))
    fora = np.array([1 - len(np.unique(l)) / n_teste for l in sorteios])
    print(f"n = {n_teste:5d}: fracao media fora da bolsa = {fora.mean():.4f}")
print(f"limite 1/e            = {np.exp(-1):.4f}")

---
## 8. Escolhendo $m$: a floresta é um botão só

O parâmetro que separa a floresta do *bagging* é o `max_features`. A regra empírica
é $m \approx p/3$ em regressão (e $\sqrt p$ em classificação), mas é um
hiperparâmetro como qualquer outro. Vamos varrer e olhar o risco junto com o $\rho$
que a Seção 6 ensinou a medir.

In [ ]:
linhas = []
for m in [1, 2, 3, 4, 6]:
    rf = RandomForestRegressor(n_estimators=300, max_features=m, oob_score=True,
                               random_state=0, n_jobs=-1).fit(X_p, y_p)
    linhas.append({"m": m, "m/p": m / d_p,
                   "risco (teste)": np.mean((rf.predict(X_pte) - r_pte) ** 2),
                   "R^2 OOB": rf.oob_score_})
pd.DataFrame(linhas).set_index("m").round(4)

O mínimo fica no meio, como a teoria manda: $m$ pequeno demais descorrelaciona bem
mas estraga cada árvore (o $v$ sobe mais do que o $\rho$ desce); $m = p$ é o
*bagging*, com $\rho$ mais alto. A regra $m \approx p/3$ não é lei — aqui o melhor
foi $m = p/2$ — mas cai perto o bastante para servir de ponto de partida.

E note que o $R^2$ OOB aponta **o mesmo vencedor** que o risco de teste, e põe o
$m=1$ em último — tudo isso sem gastar uma única dobra de validação cruzada. No
meio da tabela as duas ordens não coincidem exatamente, o que é esperado: as
diferenças ali são pequenas perto do ruído das duas medidas. Para escolher
`max_features` numa floresta, o OOB costuma bastar.

Com muito lixo na lista, sortear poucas variáveis por nó aumenta a chance de
nenhuma delas prestar. Acrescentamos 10 colunas de puro ruído, indo de $p=6$ para
$p=16$, e varremos $m$ nos dois casos.

In [ ]:
for extras in (0, 10):
    rng_m = np.random.default_rng(5)
    d_tot = d_p + extras
    Xm = rng_m.uniform(-2, 2, size=(n_p, d_tot))
    ym = alvo_p(Xm) + rng_m.normal(0, 1.0, size=n_p)
    Xm_te = rng_m.uniform(-2, 2, size=(4000, d_tot))
    rm_te = alvo_p(Xm_te)

    linhas_m = []
    for m in [1, 2, 3, 4, 6, 8, 12, 16]:
        if m > d_tot:
            continue
        rf_m = RandomForestRegressor(n_estimators=300, max_features=m, oob_score=True,
                                     random_state=0, n_jobs=-1).fit(Xm, ym)
        linhas_m.append({"m": m, "m/p": m / d_tot,
                         "risco (teste)": np.mean((rf_m.predict(Xm_te) - rm_te) ** 2),
                         "R^2 OOB": rf_m.oob_score_})
    t = pd.DataFrame(linhas_m).set_index("m")
    print(f"p = {d_tot}  ({extras} colunas irrelevantes)   m* = {t['risco (teste)'].idxmin()}")
    print(t.round(4).to_string())
    print()

**O $m$ ótimo se desloca para a direita, e bastante:** de $m^*=3$ com $p=6$ para
$m^*=12$ com $p=16$. Em fração das colunas, de $0{,}50$ para $0{,}75$ — bem acima
do $p/3$ da regra empírica, que daria $2$ e $5$.

O mecanismo é o que o enunciado antecipava. Com $p=16$ e só 3 colunas úteis,
sortear $m=1$ dá $3/16$ de chance de o nó ter alguma variável que preste; o resto
do tempo a árvore corta em ruído. O risco com $m=1$ é $1{,}53$, contra $0{,}99$ no
ótimo — 55% pior.

Vale notar o que **não** mudou: em ambos os casos, usar todas as colunas ($m=p$) é
pior que o ótimo. Mesmo afogada em lixo, a floresta ainda quer alguma
descorrelação entre as árvores; ela só quer menos. O `max_features` continua sendo
um botão a girar, e a regra $p/3$ é um chute inicial, não uma resposta — o que
fica mais evidente justamente quando há colunas irrelevantes, que é a situação
comum fora do livro.

Já o $R^2$ OOB, que sai de graça junto com o ajuste, acerta o mínimo com $p=6$ e
**erra por um degrau** com $p=16$: ele aponta $m=8$, onde o risco verdadeiro vale
$1{,}0134$, contra $0{,}9899$ em $m=12$. São 2,4% de diferença, e as duas curvas
são rasas nessa faixa. É a mesma história da Aula 03 — o estimador barato acerta a
região, não o ponto —, e por isso ele serve para escolher entre $m=1$ e $m=8$, não
entre $m=8$ e $m=12$.

---
## 9. *Boosting*: o sentido inverso do eixo

O *bagging* parte de árvores de **viés baixo e variância alta** e usa a média para
derrubar a variância. O *boosting* faz o contrário: parte de $g \equiv 0$ — viés
máximo, variância zero — e vai comprando viés de volta em prestações de tamanho
$\lambda$, cada uma ajustada aos **resíduos** do que já existe.

Os dois percorrem o mesmo eixo da Aula 01 em sentidos opostos, e por isso os riscos
são opostos: excesso de *bagging* é inofensivo, excesso de *boosting* superajusta.
Vamos ver os dois lado a lado, num problema pequeno e ruidoso.

In [ ]:
rng_e = np.random.default_rng(12)
n_e, d_e, ruido = 150, 8, 1.5
X_e = rng_e.uniform(-2, 2, size=(n_e, d_e))
y_e = alvo_p(X_e) + rng_e.normal(0, ruido, size=n_e)
X_ete = rng_e.uniform(-2, 2, size=(4000, d_e))
y_ete = alvo_p(X_ete) + rng_e.normal(0, ruido, size=4000)

Bs_e = np.array([1, 2, 3, 5, 8, 12, 20, 35, 60, 100, 175, 300, 500])
rf_te, rf_tr = [], []
for B in Bs_e:
    m = RandomForestRegressor(n_estimators=int(B), max_features=1 / 3,
                              random_state=0).fit(X_e, y_e)
    rf_te.append(mean_squared_error(y_ete, m.predict(X_ete)))
    rf_tr.append(mean_squared_error(y_e, m.predict(X_e)))

gb = GradientBoostingRegressor(learning_rate=0.05, n_estimators=1500,
                               max_depth=3, random_state=0).fit(X_e, y_e)
gb_te = np.array([mean_squared_error(y_ete, p) for p in gb.staged_predict(X_ete)])
gb_tr = np.array([mean_squared_error(y_e, p) for p in gb.staged_predict(X_e)])
passos = np.arange(1, len(gb_te) + 1)
B_melhor = passos[int(np.argmin(gb_te))]

print(f"floresta: risco em B=100 -> {rf_te[9]:.4f}, em B=500 -> {rf_te[-1]:.4f}"
      f"  ({100*(rf_te[-1]/rf_te[9]-1):+.1f}%)")
print(f"boosting: minimo em B={B_melhor} ({gb_te.min():.4f}), "
      f"em B=1500 -> {gb_te[-1]:.4f}  (+{100*(gb_te[-1]/gb_te.min()-1):.0f}%)")

In [ ]:
fig, (ax1, ax2) = subplots(1, 2, figsize=(7.8, 3.0), sharey=True)
ax1.plot(Bs_e, rf_te, "o-", ms=3, color="crimson", label="risco (teste)")
ax1.plot(Bs_e, rf_tr, "s--", ms=3, color="gray", label="erro de treino")
ax1.set_xscale("log"); ax1.set_ylim(0, 6)
ax1.set_xlabel("B"); ax1.set_ylabel("erro quadratico medio")
ax1.set_title("floresta: estabiliza", fontsize=9); ax1.legend(fontsize=7.5)

ax2.plot(passos, gb_te, color="crimson", label="risco (teste)")
ax2.plot(passos, gb_tr, color="gray", ls="--", label="erro de treino")
ax2.axvline(B_melhor, ls=":", color="steelblue")
ax2.set_xscale("log"); ax2.set_xlabel("B")
ax2.set_title(f"boosting: minimo em B={B_melhor}, depois sobe", fontsize=9)
ax2.legend(fontsize=7.5)

Duas curvas, duas moralidades. À esquerda, dobrar $B$ de 100 para 500 não muda
nada: na floresta, $B$ não é um hiperparâmetro a ajustar, é um orçamento de
computação. Use quantas árvores couberem no seu tempo.

À direita, o mínimo aparece muito cedo e o risco **sobe** depois — e o erro de
treino continua caindo o tempo todo, exatamente como na Aula 01. Aqui $B$ é um
hiperparâmetro de verdade, e escolher errado custa caro.

O `scikit-learn` resolve isso com *early stopping*: separe uma fração dos dados e
pare quando ela deixar de melhorar.

In [ ]:
gb_auto = GradientBoostingRegressor(learning_rate=0.05, n_estimators=1500,
                                    max_depth=3, validation_fraction=0.2,
                                    n_iter_no_change=20, random_state=0).fit(X_e, y_e)
print(f"arvores efetivamente usadas: {gb_auto.n_estimators_}  (de 1500 pedidas)")
print(f"risco no teste             : "
      f"{mean_squared_error(y_ete, gb_auto.predict(X_ete)):.4f}")
print(f"melhor possivel (B={B_melhor})    : {gb_te.min():.4f}")
print(f"sem parada  (B=1500)       : {gb_te[-1]:.4f}")

A regra clássica é "$\lambda$ pequeno e $B$ grande". Vale ver os dois extremos: um
passo dez vezes maior e um cem vezes menor que o $0{,}05$ usado acima.

In [ ]:
import time

for lr in (0.5, 0.05, 0.005):
    t0 = time.perf_counter()
    gb_lr = GradientBoostingRegressor(learning_rate=lr, n_estimators=1500,
                                      max_depth=3, random_state=0).fit(X_e, y_e)
    segundos = time.perf_counter() - t0
    te_lr = np.array([mean_squared_error(y_ete, pr) for pr in gb_lr.staged_predict(X_ete)])
    B_lr = int(np.argmin(te_lr)) + 1
    print(f"lr = {lr:<6} B* = {B_lr:4d}   risco minimo {te_lr.min():.4f}   "
          f"em B=1500 {te_lr[-1]:.4f}   1500 arvores em {segundos:.1f}s")

| `learning_rate` | $B^*$ | risco no mínimo | risco em $B=1500$ |
| --- | --- | --- | --- |
| $0{,}5$ | 3 | $3{,}574$ | $4{,}382$ |
| $0{,}05$ | 32 | $3{,}234$ | $3{,}750$ |
| $0{,}005$ | 347 | $3{,}199$ | $3{,}466$ |

**A regra se confirma, com retornos decrescentes.** Passar de $0{,}5$ para $0{,}05$
melhora o risco em 9,5% e custa dez vezes mais árvores. Passar de $0{,}05$ para
$0{,}005$ melhora 1,1% e custa mais onze vezes. O ganho é real e é pequeno.

Com $\lambda=0{,}5$ o mínimo aparece na **terceira** árvore, e a partir daí o
modelo só piora: cada passo já corrige demais, e o que sobra para as 1497 árvores
seguintes é ajustar ruído. O risco final é 22% pior que o mínimo. Com
$\lambda=0{,}005$, a curva é rasa perto do mínimo e mesmo parar tarde custa pouco —
$3{,}47$ contra $3{,}20$.

Quanto ao tempo: **aqui, nada.** Mil e quinhentas árvores levam de $0{,}3$ a
$0{,}5$ segundo nos três casos, porque com $n=150$ o custo é dominado pelo próprio
laço do Python. O preço do $\lambda$ pequeno não é o segundo a mais neste notebook,
é o $B^*$ que ele exige — $347$ contra $3$. Com $n$ grande, o tempo de ajuste é
proporcional a $B$, e aí a conta aparece.

A leitura prática: $\lambda$ pequeno compra **segurança**, não desempenho. Ele
achata a curva perto do mínimo, e é isso que torna a parada antecipada confiável.

---
## 10. Importância de variáveis: por que não confiar na padrão

Perdemos a árvore única para desenhar, e em troca ganhamos uma medida de
importância: soma-se a redução de RSS de todas as divisões que usaram cada
variável, e faz-se a média sobre as árvores. É o `feature_importances_`.

Ela tem um viés conhecido: favorece variáveis com **muitos valores distintos**,
porque elas oferecem mais cortes candidatos e ganham no sorteio mais vezes. Vamos
medir esse viés da forma mais crua possível — acrescentando ao problema duas
variáveis **completamente irrelevantes**, uma contínua e uma binária.

Repare de antemão que a nossa população só usa **três** covariáveis:
$\sin(1{,}5x_1) + 0{,}8\,x_2x_3 + 0{,}5x_1^2$. As variáveis $x_4$, $x_5$ e $x_6$ já
eram irrelevantes; vamos acrescentar mais duas, uma contínua e uma binária. Cinco
colunas inúteis, e uma medida honesta deveria dar zero a todas.

In [ ]:
def montar(n_obs, semente):
    rng_i = np.random.default_rng(semente)
    M = rng_i.uniform(-2, 2, size=(n_obs, d_p))
    alvo = alvo_p(M) + rng_i.normal(0, 1.0, size=n_obs)
    M = np.hstack([M,
                   rng_i.uniform(-2, 2, size=(n_obs, 1)),          # lixo continuo
                   rng_i.integers(0, 2, size=(n_obs, 1)).astype(float)])  # lixo binario
    return M, alvo


nomes = [f"x{j+1}" for j in range(d_p)] + ["lixo continuo", "lixo binario"]
X_i, y_i = montar(800, 2)
X_h, y_h = montar(3000, 77)          # dados que a floresta nunca viu

rf_i = RandomForestRegressor(n_estimators=400, max_features=1/3,
                             random_state=0, n_jobs=-1).fit(X_i, y_i)

comp = pd.DataFrame({
    "impureza": rf_i.feature_importances_,
    "permutacao (treino)": permutation_importance(
        rf_i, X_i, y_i, n_repeats=20, random_state=0).importances_mean,
    "permutacao (teste)": permutation_importance(
        rf_i, X_h, y_h, n_repeats=20, random_state=0).importances_mean,
}, index=nomes)
comp.round(4)

In [ ]:
fig, ax = subplots(figsize=(6.4, 3.0))
pos = np.arange(len(nomes))
for desloc, coluna, cor in [(-0.26, "impureza", "crimson"),
                            (0.0, "permutacao (treino)", "darkorange"),
                            (0.26, "permutacao (teste)", "steelblue")]:
    v = comp[coluna].clip(lower=0)
    ax.bar(pos + desloc, v / v.sum(), width=0.25, label=coluna, color=cor)
ax.set_xticks(pos); ax.set_xticklabels(nomes, rotation=30, ha="right", fontsize=7)
ax.set_ylabel("importancia relativa"); ax.legend(fontsize=7.5)

inuteis = ["x4", "x5", "x6", "lixo continuo", "lixo binario"]
print("credito dado as 5 covariaveis INUTEIS:")
for coluna in comp.columns:
    print(f"   {coluna:22s}: {comp.loc[inuteis, coluna].sum() / comp[coluna].clip(lower=0).sum():6.1%}")
print(f"\nlixo continuo x lixo binario, por impureza: "
      f"{comp.loc['lixo continuo', 'impureza'] / comp.loc['lixo binario', 'impureza']:.0f}x")

> **A lição.** Duas leituras, e a segunda é a que quase ninguém conta.
>
> **A importância por impureza distribui crédito para quem não fez nada.** As cinco
> covariáveis inúteis levam uma fatia enorme do total — e entre as duas de lixo, a
> contínua recebe várias vezes mais que a binária. A diferença entre elas é
> puramente a **cardinalidade**: a contínua oferece centenas de cortes candidatos, a
> binária oferece um. Nenhuma das duas tem relação alguma com $y$.
>
> **A importância por permutação só funciona fora do treino.** Calculada nos
> próprios dados de ajuste, ela repete o vício: a floresta *decorou* ruído usando a
> variável contínua, então embaralhá-la piora o ajuste de treino. Calculada em dados
> que o modelo nunca viu, as cinco inúteis caem para praticamente zero — inclusive
> negativas, que é o ruído da medida em torno de zero.
>
> Regra prática: `feature_importances_` só para ordenar grosseiramente;
> `permutation_importance` **sempre em dados separados**.

---
## 11. Os quatro no `superconductivity.csv`

Fechamos com a comparação completa, no conjunto que já usamos nas Aulas 02 a 04:
21.263 materiais, 81 atributos, temperatura crítica como resposta.

In [ ]:
import os

_nome = "superconductivity.csv"

# procura em dois lugares, sem baixar nada da internet: a pasta deste
# notebook primeiro ou então ../../recursos/dados/
_lugares = [_nome, os.path.join("..", "..", "recursos", "dados", _nome)]
_caminho = next((c for c in _lugares if os.path.exists(c)), None)

if _caminho is None:
    raise FileNotFoundError(
        f"nao encontrei '{_nome}'. Procurei nesta pasta e em "
        "../../recursos/dados/. Ponha o .csv ao lado deste notebook, "
        "ou mude o caminho se for necessário."
    )

df = pd.read_csv(_caminho)
Xs = df.drop(columns="critical_temp")
ys = df["critical_temp"].values

rng_s = np.random.default_rng(0)
sub = rng_s.choice(len(ys), size=6000, replace=False)
X_tr, X_te, y_tr, y_te = skm.train_test_split(Xs.values[sub], ys[sub],
                                              test_size=0.3, random_state=0)
print("treino:", X_tr.shape, "  teste:", X_te.shape)

In [ ]:
import time

modelos = {
    "Ridge (Aula 02)": Pipeline([("escala", StandardScaler()),
                                 ("ridge", skl.Ridge(alpha=1.0))]),
    "arvore podada": DecisionTreeRegressor(ccp_alpha=1.0, random_state=0),
    "bagging (B=300)": BaggingRegressor(DecisionTreeRegressor(random_state=0),
                                        n_estimators=300, random_state=0, n_jobs=-1),
    "floresta (B=300, m=d/3)": RandomForestRegressor(n_estimators=300,
                                                     max_features=1/3,
                                                     random_state=0, n_jobs=-1),
    "boosting (parada antecipada)": GradientBoostingRegressor(
        learning_rate=0.05, n_estimators=2000, max_depth=5,
        validation_fraction=0.2, n_iter_no_change=50, random_state=0),
}

linhas = []
for nome, m in modelos.items():
    t0 = time.perf_counter()
    m.fit(X_tr, y_tr)
    linhas.append({"modelo": nome,
                   "EQM no teste": mean_squared_error(y_te, m.predict(X_te)),
                   "segundos": time.perf_counter() - t0})
linhas.append({"modelo": "chutar a media", "EQM no teste": y_te.var(), "segundos": 0.0})
pd.DataFrame(linhas).set_index("modelo").round(3)

A distância entre o modelo linear e os *ensembles* é enorme: a Ridge fica em mais
do dobro do erro da floresta. Quatro observações sobre o resto.

A **árvore isolada** perde para tudo — mas ganha da Ridge, o que já diz alguma
coisa sobre a não linearidade deste problema. Ela é matéria-prima, não produto
final.

O **bagging e a floresta empatam**: as 81 colunas
são fortemente redundantes, então não existe *uma* covariável dominante para todas
as árvores agarrarem. O $\rho$ do *bagging* já nasce baixo aqui, e sobra pouco para
a floresta reduzir — ao contrário do problema sintético da Seção 6, onde $x_1$
dominava.

O ***boosting*** **não ganha** nesta configuração, e é de longe o mais caro. Vale
registrar isso contra o folclore de que *boosting* sempre vence: com esta amostra e
estes hiperparâmetros, a floresta entrega o mesmo ou melhor por um décimo do tempo.
Um *boosting* bem ajustado provavelmente passa à frente — mas "bem ajustado" é
justamente o trabalho que a floresta não exige.

E é aí que está a assimetria prática entre os dois. A floresta tem um
hiperparâmetro que importa (`max_features`), tolera $B$ grande e ainda entrega o
OOB de graça. O *boosting* tem três que interagem ($\lambda$, profundidade, $B$) e
pune quem erra. Como primeiro modelo não linear a experimentar, a floresta é a
escolha de menor arrependimento.

In [ ]:
rf_final = modelos["floresta (B=300, m=d/3)"]
imp = pd.Series(rf_final.feature_importances_, index=Xs.columns)
print(imp.sort_values(ascending=False).head(8).round(4).to_string())
print(f"\nas 8 mais importantes somam {imp.sort_values().iloc[-8:].sum():.1%} do total")
print(f"as 40 menos importantes somam {imp.sort_values().iloc[:40].sum():.1%}")

A caixa da Seção 10 avisa que importâncias de floresta são instáveis quando as
colunas são correlacionadas — e as 81 daqui são grupos inteiros de medidas da mesma
propriedade. Vale medir o que se perde ao acreditar na lista.

In [ ]:
top10 = np.argsort(rf_final.feature_importances_)[::-1][:10]
print("as 10 mais importantes:", list(Xs.columns[top10][:4]), "...")
print(f"elas somam {imp.sort_values().iloc[-10:].sum():.1%} da importancia total\n")

t0 = time.perf_counter()
rf_10 = RandomForestRegressor(n_estimators=300, max_features=1 / 3,
                              random_state=0, n_jobs=-1).fit(X_tr[:, top10], y_tr)
seg_10 = time.perf_counter() - t0
eqm_10 = mean_squared_error(y_te, rf_10.predict(X_te[:, top10]))
eqm_81 = mean_squared_error(y_te, rf_final.predict(X_te))

print(f"81 colunas: EQM {eqm_81:.2f}")
print(f"10 colunas: EQM {eqm_10:.2f}   ({100 * (eqm_10 - eqm_81) / eqm_81:+.1f}%)"
      f"   ajuste em {seg_10:.2f}s")

Jogar fora 71 colunas custa **13% de EQM** — de $122{,}9$ para $139{,}1$ — e devolve
um ajuste cerca de duas vezes mais rápido. Se a troca compensa depende do uso; se
a lista merece confiança, não.

E não merece, pelo motivo exato da caixa da Seção 10. As 81 colunas são
estatísticas-resumo das mesmas propriedades atômicas: dez versões de massa
atômica, várias de condutividade térmica. Quando duas colunas carregam quase a
mesma informação, a floresta reparte a importância entre elas de forma
essencialmente arbitrária — quem foi sorteada primeiro leva o crédito. Uma segunda
floresta, com outra semente, produz outro ranking, e a diferença entre a décima e a
décima primeira posição não significa nada.

As dez primeiras somam $63{,}1\%$ da importância total. Isso quer dizer que as 71
descartadas ainda respondem por mais de um terço — e os 13% de piora medem
exatamente isso: elas não eram redundantes a ponto de sair de graça.

A conduta defensável não é confiar no ranking, é **medir o que se perde**, que é o
que esta célula faz. Se 13% for aceitável para o seu problema, corte; se não, fique
com as 81 e pague o dobro do tempo.

---
## Resumo

| Conceito | Onde apareceu | O que vimos |
|---|---|---|
| árvore = partição | §2 | os dois ramos cortam $x_2$ em pontos diferentes: interação de graça |
| cortes nos eixos | §2 | fronteira diagonal vira escadinha, e cada degrau custa dados |
| miopia gananciosa | §3 | 4 folhas resolvem o XOR; a gananciosa gasta 12, e uma parada precoce a mata na raiz |
| poda | §4 | `cost_complexity_pruning_path` dá a sequência inteira; $\alpha$ sai por CV |
| instabilidade | §5 | a variável da raiz nunca muda (há uma dominante), mas o corte passeia por 1/4 do domínio |
| $\Var(\bar g) = \rho v + (1-\rho)v/B$ | §6 | a fórmula bate com o medido; o piso $\rho v$ é real |
| floresta | §6, §8 | derruba $\rho$ à custa de subir $v$ — e o produto compensa |
| OOB | §7 | estimativa de graça, próxima do teste; a fração de fora é $1/e$ |
| $B$ na floresta × no boosting | §9 | inofensivo de um lado, superajuste do outro |
| importância por impureza | §10 | dá crédito a 5 covariáveis inúteis; a permutação só corrige isso **fora do treino** |
| caso real | §11 | floresta e *bagging* empatam na frente; o *boosting* não ganha e custa 10× mais |

**Leitura recomendada.** [AME] §4.8 (árvores), §4.9 (*bagging* e florestas) e §4.10
(*boosting*). [ISLP] Capítulo 8 inteiro: §8.1 (crescer e podar, com a mesma figura
de partição da Seção 2), §8.2.1–8.2.2 (*bagging*, OOB e importâncias), §8.2.3
(florestas) e §8.2.4 (*boosting*, com a discussão dos três hiperparâmetros).

**Para praticar.** `Lista teorica 05.pdf` (teórica, com gabarito) e
`Lista prática 05.ipynb` (prática, para completar as lacunas), nesta mesma
pasta.

**A seguir.** A Aula 06 arruma a casa: `Pipeline`, `ColumnTransformer` e a
disciplina que impede que a padronização, a seleção de variáveis ou a imputação
vazem informação do teste para o treino. Sem ela, todos os números deste notebook
seriam suspeitos.